#Installing the required libraries

In [ ]:
!%pip install torch torchvision librosa kymatio numpy

  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/12/1b/a61ce2004f9ab0ea8964a6e6168133a127795667639e2ff4f8f2bdb16a65/torch-2.12.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/19/aa/929b358b1a643849b81ec95569938044cc37dc65ab10c84eb6d82fe1bfbb/torchvision-0.27.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for sympy>=1.13.3 from https://files.pythonhosted.org/packages/a2/09/77d55d46fd61b4a135c444fc97158ef34a095e5681d0a6c10b75bf356191/sympy-1.14.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/123.0 MB 1.7 MB/s eta 0:01:11
   ---------------------------------------- 0.5/123.0 MB 4.7 MB/s eta 0:00:27
   ---------------------------------------- 0.8/123.0 MB 5.6 MB/s eta 0:00:22
   ---------------------------------------- 1.2/123.0 MB 6.1 MB/s eta

#Importing the required libraries

In [ ]:

import os
import json
import random
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import butter, sosfilt, medfilt
from torchvision.models import (
    vgg16, vgg19, densenet121, efficientnet_b2,
    inception_v3, mobilenet_v2, resnet50
)
from kymatio.numpy import Scattering1D
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import hashlib
import matplotlib.pyplot as plt

#Defining constants

In [ ]:

SR          = 4000  #Sampling Rate
DURATION    = 5.0   #Standardization to 5 seconds (20000 samples)
TARGET_LEN  = int(SR * DURATION)   # 20000 samples

#Bandpass and Media Filtering
BP_LOW      = 20
BP_HIGH     = 900
BP_ORDER    = 4  #The sharpness of the filter
NOISE_ALPHA = 0.005

#The kernel size for a Median Filter
MED_K       = 2
# Epsilon, a tiny number used to prevent a "divide by zero" error.
EPS         = 1e-9

#Wavelet Scattering Transform
WST_J       = 5  #number octaves the wavelets will look at
WST_Q       = 2  #number of wavelets per octave

#MFCC Feature Extraction
MFCC_N_MELS = 60
MFCC_FMIN   = 20
MFCC_FMAX   = 1500
MFCC_FFT    = 512
MFCC_HOP    = 48
MFCC_WIN    = 192

#Computer Vision Dimensions
IMG_H       = 256
IMG_W       = 256

#Deep Learning Hyperparameters
BATCH_SIZE  = 16
EPOCHS      = 100
LR          = 1e-4
L2          = 1e-4
PATIENCE    = 15

# Yaseen 5 classes
CLASS_NAMES = ["AS", "MR", "MS", "MVP", "N"]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

# Yaseen root folder path
DATASET_PATH = "/content/drive/MyDrive/heart_sounds"

#Audio Preprocessing


In [ ]:
#This function is used to construct the bandpass filter
def _butter_bandpass_sos(low, high, fs, order=4):
    nyq    = fs / 2.0
    low_n  = low  / nyq
    high_n = high / nyq
    return butter(order, [low_n, high_n], btype="band", output="sos")

#Butterworth Bandpass Filter
_BPF_SOS = _butter_bandpass_sos(BP_LOW, BP_HIGH, SR, BP_ORDER)

#This function implements the audio preprocessing pipeline
def load_and_preprocess(audio_path: str) -> np.ndarray:
    signal, _ = librosa.load(audio_path, sr=SR, mono=True, duration=DURATION)

    # Amplitude normalisation
    max_amp = np.max(np.abs(signal))
    if max_amp > 0:
        signal = signal / (max_amp + EPS)

    # Bandpass filter
    signal = sosfilt(_BPF_SOS, signal)

    # Adaptive noise gate
    thr = NOISE_ALPHA * np.max(np.abs(signal))
    signal = np.where(np.abs(signal) >= thr, signal, 0.0)

    # NaN/Inf correction
    signal = np.where(np.isfinite(signal), signal, 0.0)

    # Median filter
    signal = medfilt(signal, kernel_size=2 * MED_K + 1)

    # Final renormalisation
    max_amp = np.max(np.abs(signal))
    if max_amp > 0:
        signal = signal / (max_amp + EPS)
    else:
        signal = np.zeros_like(signal)

    # Trailing silence removal
    nonzero_idx = np.where(np.abs(signal) > 1e-6)[0]
    if len(nonzero_idx) > 0:
        signal = signal[: nonzero_idx[-1] + 1]

    # Pad or truncate to TARGET_LEN
    if len(signal) < TARGET_LEN:
        signal = np.pad(signal, (0, TARGET_LEN - len(signal)))
    else:
        signal = signal[:TARGET_LEN]

    return signal.astype(np.float32)


#Feature Extraction

In [ ]:
#This function computes the normalized log-mel spectrogram
def extract_mfcc(signal: np.ndarray) -> np.ndarray:
    mel_spec = librosa.feature.melspectrogram(
        y=signal, sr=SR,
        n_mels=MFCC_N_MELS, fmin=MFCC_FMIN, fmax=MFCC_FMAX,
        n_fft=MFCC_FFT, hop_length=MFCC_HOP, win_length=MFCC_WIN,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    p1  = np.percentile(log_mel, 1)
    p99 = np.percentile(log_mel, 99)
    log_mel = np.clip(log_mel, p1, p99)
    denom = (log_mel.max() - log_mel.min()) + EPS
    return ((log_mel - log_mel.min()) / denom).astype(np.float32)


#This function computes the Wavelet Scattering Transform
def extract_wst(signal: np.ndarray) -> np.ndarray:
    scattering = Scattering1D(J=WST_J, shape=TARGET_LEN, Q=WST_Q)
    Sx = scattering(signal)

    row_std = Sx.std(axis=1)
    Sx = Sx[row_std > 1e-10]
    Sx = np.log1p(np.abs(Sx))

    mu, sigma = Sx.mean(), Sx.std()
    if sigma > 0:
        Sx = (Sx - mu) / sigma
    return Sx.astype(np.float32)

#This function performs feature fusing
def fuse_wst_mfcc(wst: np.ndarray, mfcc: np.ndarray) -> np.ndarray:
  def _bicubic_resize(arr):
      t = torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(0)
      t = F.interpolate(t, size=(IMG_H, IMG_W), mode="bicubic",
                        align_corners=False)
      return t.squeeze()

  def _zscore(t):
      mu, sigma = t.mean(), t.std()
      return (t - mu) / (sigma + EPS)

  fused = (_zscore(_bicubic_resize(wst)) + _zscore(_bicubic_resize(mfcc))) / 2.0
  fused = torch.clamp(fused, -3.0, 3.0)
  f_min, f_max = fused.min(), fused.max()
  return ((fused - f_min) / ((f_max - f_min) + EPS)).numpy().astype(np.float32)


#This function extracts the features of the audio signal that receives as an input
def compute_feature(wav_path: str) -> np.ndarray:
    """Compute fused feature for one file."""
    signal = load_and_preprocess(wav_path)
    wst    = extract_wst(signal)
    mfcc   = extract_mfcc(signal)
    fused  = fuse_wst_mfcc(wst, mfcc)
    np.save(cp, fused)
    return fused



#Bulding the dataset


In [ ]:

# Dataset class
class HeartDataset(Dataset):
    def __init__(self, samples):
        # samples: list of (fused_array (H,W), label_int)
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feat, label = self.samples[idx]
        feat_t = torch.from_numpy(feat).float().unsqueeze(0).expand(3, -1, -1)
        return feat_t, torch.tensor(label, dtype=torch.long)



def scan_yaseen_dataset(root_dir: str):
    """
    Finds all .wav files and pairs them with their class numbers.
    Works if the class folders are directly in the root folder,
    or if they are located inside subfolders.
    """
    entries = []

    # Flat structure
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(root_dir, class_name)
        if os.path.isdir(class_dir):
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(".wav"):
                    entries.append((
                        os.path.join(class_dir, fname),
                        CLASS_TO_IDX[class_name]
                    ))

    # Nested Structure
    if not entries:
        for subset in os.listdir(root_dir):
            subset_path = os.path.join(root_dir, subset)
            if not os.path.isdir(subset_path):
                continue
            for class_name in CLASS_NAMES:
                class_dir = os.path.join(subset_path, class_name)
                if os.path.isdir(class_dir):
                    for fname in os.listdir(class_dir):
                        if fname.lower().endswith(".wav"):
                            entries.append((
                                os.path.join(class_dir, fname),
                                CLASS_TO_IDX[class_name]
                            ))

    if not entries:
        raise RuntimeError(
            f"No wav files found under {root_dir}. "
            f"Expected subfolders: {CLASS_NAMES}"
        )

    print(f"Found {len(entries)} wav files total.")
    counts = np.bincount([e[1] for e in entries], minlength=NUM_CLASSES)
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name}: {counts[i]}")
    return entries


def build_datasets(root_dir: str):
    """
    1. Scan all wav files with labels.
    2. Stratified split 80/10/10 by file .
    3. Compute features .
    Returns train_ds, val_ds, test_ds.
    """
    entries = scan_yaseen_dataset(root_dir)
    paths   = [e[0] for e in entries]
    labels  = [e[1] for e in entries]

    # Stratified 80 / 20 split first
    tr_paths, tmp_paths, tr_labels, tmp_labels = train_test_split(
        paths, labels,
        test_size=0.20,
        stratify=labels,
        random_state=42
    )
    # Then the 20% is split into validation (10%) and testing (10%)
    val_paths, te_paths, val_labels, te_labels = train_test_split(
        tmp_paths, tmp_labels,
        test_size=0.50,
        stratify=tmp_labels,
        random_state=42
    )


    def _load_split(path_list, label_list, split_name):
          features, labs = [], []
          print(f"\nComputing {split_name} features ({len(path_list)} files)…")
          for i, (p, l) in enumerate(zip(path_list, label_list)):
              try:
                  feat = compute_feature(p)
                  features.append(feat)
                  labs.append(l)
              except Exception as e:
                  print(f"  [WARN] skipping {p}: {e}")
              if (i + 1) % 50 == 0:
                  print(f"  {i+1}/{len(path_list)} done")
          return features, labs

    tr_feats,  tr_labs  = _load_split(tr_paths,  tr_labels,  "train")
    val_feats, val_labs = _load_split(val_paths, val_labels, "val")
    te_feats,  te_labs  = _load_split(te_paths,  te_labels,  "test")

    print(f"\nSplit sizes — train: {len(tr_labs)}, "
            f"val: {len(val_labs)}, test: {len(te_labs)}")

    # Class distribution check
    for split_name, labs in [("train", tr_labs), ("val", val_labs), ("test", te_labs)]:
          counts = np.bincount(labs, minlength=NUM_CLASSES)
          print(f"  {split_name} dist: {dict(zip(CLASS_NAMES, counts))}")

    train_ds = HeartDataset(list(zip(tr_feats,  tr_labs)))
    val_ds   = HeartDataset(list(zip(val_feats, val_labs)))
    test_ds  = HeartDataset(list(zip(te_feats,  te_labs)))
    return train_ds, val_ds, test_ds







#Model Building


In [ ]:
 #This is a helper method that builds the classifier head
def _build_classifier_head(in_features: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_features, 1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(1024, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.25),
        nn.Linear(512, NUM_CLASSES),
    )


#This is the method that is used to build the models
def build_model(name: str) -> nn.Module:
    name = name.lower()
    if name == "vgg16":
        m = vgg16(weights="IMAGENET1K_V1")
        m.classifier = _build_classifier_head(m.classifier[0].in_features)
    elif name == "vgg19":
        m = vgg19(weights="IMAGENET1K_V1")
        m.classifier = _build_classifier_head(m.classifier[0].in_features)
    elif name == "densenet121":
        m = densenet121(weights="IMAGENET1K_V1")
        m.classifier = _build_classifier_head(m.classifier.in_features)
    elif name == "efficientnet_b2":
        m = efficientnet_b2(weights="IMAGENET1K_V1")
        m.classifier = _build_classifier_head(m.classifier[1].in_features)
    elif name == "inception_v3":
        m = inception_v3(weights="IMAGENET1K_V1", aux_logits=False)
        m.fc = _build_classifier_head(m.fc.in_features)
    elif name == "mobilenet_v2":
        m = mobilenet_v2(weights="IMAGENET1K_V1")
        m.classifier = _build_classifier_head(m.classifier[1].in_features)
    elif name in ("resnet50v2", "xception"):
        m = resnet50(weights="IMAGENET1K_V2")
        m.fc = _build_classifier_head(m.fc.in_features)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m

#Model Training

In [ ]:
#This model performs model training
def train_model(model, train_loader, val_loader, device,
                epochs=EPOCHS, lr=LR, patience=PATIENCE, save_path=None):

    criterion  = nn.CrossEntropyLoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=L2)

    best_val_acc, best_state, patience_count = 0.0, None, 0
    model.to(device)

    # HISTORY STORAGE
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(1, epochs + 1):

        # TRAINING
        model.train()
        running_loss = 0.0
        correct_train, total_train = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            if hasattr(outputs, "logits"):
                outputs = outputs.logits

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc  = correct_train / total_train


        # VALIDATION
        model.eval()
        val_loss = 0.0
        correct, total = 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                if hasattr(outputs, "logits"):
                    outputs = outputs.logits

                val_loss += criterion(outputs, labels).item()

                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_loss = val_loss / len(val_loader)
        val_acc  = correct / total


        # SAVE HISTORY
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)


        # PRINT
        print(f"Epoch {epoch:3d}/{epochs} | "
              f"train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | "
              f"train_acc={train_acc:.4f} | "
              f"val_acc={val_acc:.4f}")


        # EARLY STOPPING AND SAVE
        if val_acc > best_val_acc:
            best_val_acc   = val_acc
            best_state     = {k: v.cpu().clone() for k, v in
                              model.state_dict().items()}
            patience_count = 0

            if save_path is not None:
                torch.save({
                    "model_state_dict": best_state,
                    "val_acc": best_val_acc
                }, save_path)
                print(f" Saved best model to {save_path}")

        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"  Early stopping (best val_acc={best_val_acc:.4f})")
                break


    # LOAD BEST MODEL
    if best_state:
        model.load_state_dict(best_state)

    return model, history


#Model Evaluation


In [ ]:
def evaluate(model, dataloader, device, name="Test"):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(inputs.to(device))
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_targets.extend(labels.numpy())

    all_preds, all_targets = np.array(all_preds), np.array(all_targets)
    acc  = (all_preds == all_targets).mean()
    f1   = f1_score(all_targets, all_preds, average="macro",    zero_division=0)
    prec = precision_score(all_targets, all_preds, average="macro", zero_division=0)
    rec  = recall_score(all_targets, all_preds, average="macro",    zero_division=0)
    cm   = confusion_matrix(all_targets, all_preds)

    print(f"\n{'='*50}\n  {name}\n{'='*50}")
    print(f"  Accuracy : {acc:.4f}  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}  F1 Macro : {f1:.4f}")
    print(f"  Classes  : {CLASS_NAMES}")
    print(cm)
    return dict(acc=acc, precision=prec, recall=rec, f1_macro=f1, confusion=cm)



#Graph Visualizations


In [ ]:
#This function was used to plot two graphs: training and validation
#loss and traing and validation accuracy across epochs

def plot_history(history, title="Model"):
    epochs = range(1, len(history["train_loss"]) + 1)

    # Loss
    plt.figure()
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} Loss")
    plt.legend()
    plt.show()

    # Accuracy
    plt.figure()
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["val_acc"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title} Accuracy")
    plt.legend()
    plt.show()

#Main

In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print("\n[1/4] Building Yaseen datasets …")
    train_ds, val_ds, test_ds = build_datasets(DATASET_PATH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    print(f"  Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")


    histories = {}

    ARCH_NAMES = [
        "vgg16", "vgg19", "densenet121", "efficientnet_b2",
        "mobilenet_v2", "resnet50v2",
    ]

    print("\n[2/4] Training models …")
    trained_models, individual_results = [], {}

    for arch in ARCH_NAMES:
      print(f"\n  ── {arch.upper()} ──")
      model = build_model(arch)
      save_path = f"/content/drive/MyDrive/models_new/{arch}_best.pth"
      model, history = train_model(
          model,
          train_loader,
          val_loader,
          device,
          save_path=save_path #The best weights of each model are saved
      )
      histories[arch] = history

      # Evaluate on TRAIN SET
      evaluate(model, train_loader, device, name=f"{arch.upper()} (TRAIN)")
      plot_history(history, arch.upper())

      # Evaluate on TEST SET
      res = evaluate(model, test_loader, device, name=arch.upper())

      trained_models.append(model)
      individual_results[arch] = res

      #Print the summary of performance metrics
      print("\n[4/4] Summary")
      print(f"{'Model':<20} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
      print("-" * 46)
      for arch in ARCH_NAMES:
        r = individual_results[arch]
        print(f"{arch:<20} {r['acc']:>6.4f} {r['precision']:>6.4f} "
              f"{r['recall']:>6.4f} {r['f1_macro']:>6.4f}")